# histogram_tuned_classic_control

DQN and Agent0, with and without HL-Gauss at each variant's best config from
`histogram_ln_notarget_classic_control`, 100 seeds each.

In [ ]:
# Import the experiment config, the results loader and the analysis functions.
import dataclasses

import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

from analysis.curves import return_curve, subsample
from analysis.stats import mean_ci
from main import load_results

from config import ENVIRONMENTS, EXPERIMENT

PLOTS_DIR = EXPERIMENT.results_dir.parent / "plots"

In [ ]:
# Load each agent's stored runs per environment, skipping any not stored yet. A
# component's table also keeps runs from its earlier configs (e.g. Agent0 + HL's
# 10-seed picks), so only runs at its current agent hypers are loaded.
# Agent label -> component prefix.
AGENTS = {
    "DQN": "dqn",
    "Agent0": "agent0",
    "DQN + HL": "dqn_hl",
    "Agent0 + HL": "dqn_hl_ln_notarget",
}

results = {}
for agent, prefix in AGENTS.items():
    for environment in ENVIRONMENTS:
        name = f"{prefix}_{environment}"
        hypers = dataclasses.asdict(EXPERIMENT.component(name).config.AGENT_HYPERS)
        where = {f"AGENT_HYPERS.{field}": value for field, value in hypers.items()}
        runs = load_results(EXPERIMENT, name, where=where)
        if runs is not None:
            results[agent, environment] = runs

In [ ]:
# Compute each agent's mean return over steps per environment with a 95%
# bootstrap CI.
curves = {}
for (agent, environment), runs in results.items():
    steps, returns = subsample(return_curve(runs.reward, runs.done))
    curves[agent, environment] = (steps, mean_ci(returns))

In [ ]:
# Style for the learning-curve plots. Every piece of text shares one size.
FONT_SIZE = 32
STYLE = {
    "figure.figsize": (8 * len(ENVIRONMENTS), 8),
    "font.family": "Times New Roman",
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.titlesize": FONT_SIZE,
    "axes.labelsize": FONT_SIZE,
    "xtick.labelsize": FONT_SIZE,
    "ytick.labelsize": FONT_SIZE,
    "legend.fontsize": FONT_SIZE,
    "lines.linewidth": 2.5,
    "legend.frameon": False,
}
BAND_ALPHA = 0.2
# Each base agent keeps its benchmarking color; its HL variant is the paired lighter shade.
COLORS = {
    "DQN": "tab:blue",
    "Agent0": "tab:red",
    "DQN + HL": "tab:cyan",
    "Agent0 + HL": "tab:orange",
}
# Only the left panel carries the legend, since the panels share colors.
LEGEND = {
    "loc": "center right",
    "bbox_to_anchor": (1.0, 0.35),
    "handlelength": 1,
    "handletextpad": 0.4,
}
LEGEND_MARKER = {"marker": "o", "linestyle": "none", "markersize": 14}
BOX_ASPECT = 1
# Ticks sit only at the axis ends, so the labels can tuck in between them.
LABEL_PAD = -20
XLABEL = "Time Steps"
YLABEL = "Return"
TITLES = {"mountaincar": "MountainCar", "acrobot": "Acrobot", "cartpole": "Cartpole"}
YLIMS = {"mountaincar": (-1000, 0), "acrobot": (-500, 0), "cartpole": (0, 500)}

In [ ]:
# Plot every environment's panel side by side. Each panel has its own y-axis, so
# each is labelled.
with plt.rc_context(STYLE):
    fig, axes = plt.subplots(1, len(ENVIRONMENTS), squeeze=False)
    axes = axes[0]
    for ax, environment in zip(axes, ENVIRONMENTS, strict=True):
        ax.set_box_aspect(BOX_ASPECT)
        ax.set_title(TITLES[environment])
        ax.set_xlabel(XLABEL, labelpad=LABEL_PAD)
        ax.set_ylabel(YLABEL, labelpad=LABEL_PAD)
        ax.set_ylim(YLIMS[environment])
        ax.set_yticks(YLIMS[environment])
        for agent in AGENTS:
            if (agent, environment) not in curves:
                continue
            steps, (mean, low, high) = curves[agent, environment]
            color = COLORS[agent]
            ax.fill_between(steps, low, high, color=color, alpha=BAND_ALPHA)
            ax.plot(steps, mean, color=color)
            ax.set_xticks([0, steps[-1]], ["0", f"{steps[-1] // 1000}K"])
    fig.tight_layout()
    drawn = [a for a in AGENTS if any((a, s) in curves for s in ENVIRONMENTS)]
    handles = [Line2D([], [], color=COLORS[a], **LEGEND_MARKER) for a in drawn]
    axes[0].legend(handles, drawn, **LEGEND)
figures = {"grid": fig}
plt.show()

In [ ]:
# Save every plotted figure as PDF and PNG in the plots directory.
PLOTS_DIR.mkdir(exist_ok=True)
for name, fig in figures.items():
    for ext in ("pdf", "png"):
        fig.savefig(PLOTS_DIR / f"{name}.{ext}", bbox_inches="tight", dpi=200)
print(f"saved {len(figures)} plot(s) to {PLOTS_DIR}")